## Prepare Companies House business data

In an attempt to improve the bakery name classifier further, this notebook will be preparing the Companies House business-name training data to use as an additional labelled bakery classification source.

Companies that are under the SIC code 47240 are covering the retail of bread, cakes, flour confectionary, and sugar confectionary and will be treated as positive examples of bakery names. Some closely related food retail and cafes, restaurants, and takeways will be included in the dataset as negative examples.

In [ ]:
from pathlib import Path
from getpass import getpass
from requests.auth import HTTPBasicAuth
import csv

import requests
import pandas as pd


DATA_DATE = "2026-07-28"

RAW_FOLDER= Path("../data/business/raw")
RAW_FOLDER.mkdir(parents=True, exist_ok=True)

INTERIM_FOLDER = Path("../data/business/interim")
INTERIM_FOLDER.mkdir(parents=True, exist_ok=True)

RAW_OUTPUT_PATH = (RAW_FOLDER/f"companies_house_model_data_raw_{DATA_DATE}.csv")
COMPANIES_HOUSE_TRAINING_DATA_PATH = (INTERIM_FOLDER/f"companies_house_training_names_{DATA_DATE}.csv")

API_URL = ("https://api.company-information.service.gov.uk/advanced-search/companies")

DOWNLOAD_COMPANIES_HOUSE_DATA = False

session = None

if DOWNLOAD_COMPANIES_HOUSE_DATA:
    api_key = getpass("Enter Companies House REST API key: ")
    session = requests.Session()
    session.auth = HTTPBasicAuth(api_key,"")

    print("Companies House API session created.")

else:
    print("Companies House download disabled, existing raw file will be used.")

Companies House download disabled, existing raw file will be used.


# Define SIC Categories

These selected labels are only used for training labels, they are not a definitive classification of the business or registered company.

In [2]:
SIC_CATEGORIES = {
    "47240": {"Description": "Bakery and confectionery retail", "BakeryLabel": 1},
    "47210": {"Description": "Fruit and vegetable retail", "BakeryLabel": 0},
    "47220": {"Description": "Meat retail", "BakeryLabel": 0},
    "47230": {"Description": "Fish retail", "BakeryLabel": 0},
    "47250": {"Description": "Beverage retail", "BakeryLabel": 0},
    "47290": {"Description": "Other specialised food retail", "BakeryLabel": 0},
    "56102": {"Description": "Unlicensed restaurants and cafes", "BakeryLabel": 0},
    "56103": {"Description": "Takeaway food shops and mobile food stands", "BakeryLabel": 0},
}


for sic_code, category in SIC_CATEGORIES.items():
    print(sic_code, category["BakeryLabel"], category["Description"])

47240 1 Bakery and confectionery retail
47210 0 Fruit and vegetable retail
47220 0 Meat retail
47230 0 Fish retail
47250 0 Beverage retail
47290 0 Other specialised food retail
56102 0 Unlicensed restaurants and cafes
56103 0 Takeaway food shops and mobile food stands


# Download Active Companies House records

Using the advanced-search API, active records can be retrieved under each SIC code. As the API exposes a maximum of 10,000 records, all available records are retained when there are less than 10,000 records and the first 10,000 records are downloaded when there are more,

In [3]:
def download_companies_house_dataset(sic_categories, output_path):
    #Parameters from API key Developer Hub
    page_size = 5000
    max_results_per_sic = 10000 #Limited at 10000 per SIC call however this is enough for our classifier

    output_columns = [
        "CompanyNumber",
        "CompanyName",
        "SICCodes",
        "RequestedSIC",
        "BakeryLabel",
    ]

    download_info = {}

    with output_path.open(mode="w", newline="", encoding="utf-8",) as output_file:
        writer = csv.DictWriter(output_file, fieldnames=output_columns)
        writer.writeheader()

        for sic_code, category in sic_categories.items():

            start_index = 0
            total_downloaded = 0

            while True:
                parameters = {
                    "sic_codes": sic_code,
                    "company_status": "active",
                    "size": page_size,
                    "start_index": start_index}

                response = session.get(API_URL, params=parameters, timeout=60)

                response.raise_for_status()

                response_data = response.json()

                companies = response_data.get("items", [],)

                if not companies:
                    break

                for company in companies:
                    writer.writerow({
                        "CompanyNumber": company.get("company_number"),
                        "CompanyName": company.get("company_name"),
                        "SICCodes": "|".join( company.get("sic_codes" or [])),
                        "RequestedSIC": sic_code,
                        "BakeryLabel": (category["BakeryLabel"]),
                    })

                page_count = len(companies)

                total_downloaded += page_count
                start_index += page_count

                total_available = int(response_data.get("hits", total_downloaded))
                target_total = min(total_available, max_results_per_sic)

                print(f"SIC {sic_code}: {total_downloaded} of {target_total} from the complete {total_available}")

                if total_downloaded >= target_total:
                    break

            download_info[sic_code] = total_downloaded
        
    print(f"\nRaw Companies House dataset saved to: {output_path}")

    return download_info

In [8]:
if DOWNLOAD_COMPANIES_HOUSE_DATA:
    download_info = download_companies_house_dataset(sic_categories=SIC_CATEGORIES, output_path=RAW_OUTPUT_PATH)
    print("\nSummary for Companies House raw dataset download:")

    for sic_code, company_count in (download_info.items()):
        print(f"{sic_code}: {company_count} rows")

elif RAW_OUTPUT_PATH.exists():
    print(f"Using existing Companies House raw file: {RAW_OUTPUT_PATH}")

else:
    raise FileNotFoundError("The Companies House raw file was not found, check RAW_OUTPUT_PATH or rerun with DOWNLOAD_COMPANIES_HOUSE_DATA=True")


Using existing Companies House raw file: ..\data\raw\business_data\companies_house_model_data_raw_2026-07-28.csv


# Load and inspect records before preparing for model

Downloaded records are loaded with the correct datatypes and rows with required data missing are excluded.

In [11]:
companies_house_raw = pd.read_csv(
    RAW_OUTPUT_PATH,
    dtype={
        "CompanyNumber": "string",
        "CompanyName": "string",
        "SICCodes": "string",
        "RequestedSIC": "string",
    },
)

companies_house_raw["BakeryLabel"] = pd.to_numeric(
    companies_house_raw["BakeryLabel"],
    errors="coerce",
)

companies_house_raw = (
    companies_house_raw
    .dropna(
        subset=[
            "CompanyNumber",
            "CompanyName",
            "BakeryLabel",
        ]
    )
    .copy()
)

companies_house_raw["BakeryLabel"] = (
    companies_house_raw["BakeryLabel"]
    .astype("int64")
)

print(
    f"Raw rows: "
    f"{len(companies_house_raw):,}"
)

print("\nRaw label counts:")

print(
    companies_house_raw["BakeryLabel"]
    .value_counts()
    .rename({
        0: "Non-bakery",
        1: "Bakery",
    })
)

companies_house_raw.head()

Raw rows: 56,341

Raw label counts:
BakeryLabel
Non-bakery    48453
Bakery         7888
Name: count, dtype: int64


,CompanyNumber,CompanyName,SICCodes,RequestedSIC,BakeryLabel
0,SC532505,POLIT LTD,47110|47240|47290|47789,47240,1
1,12459244,RAJA TRADING LIMITED,46170|46190|47240,47240,1
2,11691479,SIXTY ONE WALCOT LTD,47240,47240,1
3,10996639,SO SWEET CANDY STORE LIMITED,47240,47240,1
4,12243353,SO SWEET EXETER LIMITED,47240,47240,1


# Checking for and removing duplicate records

One company can be under multiple SIC codes as they may cover multiple labels. For our purpose of classification, these are too ambigious and as such will be excluded. Any companies that are consistently labelled but have duplicate records will be reduced to one row per record.

In [12]:
company_label_counts = (
    companies_house_raw
    .groupby("CompanyNumber")["BakeryLabel"]
    .nunique()
)

conflicting_company_numbers = (
    company_label_counts[
        company_label_counts > 1
    ]
    .index
)

print(
    "Companies with conflicting labels:",
    len(conflicting_company_numbers),
)

companies_house_companies = (
    companies_house_raw[
        ~companies_house_raw["CompanyNumber"]
        .isin(conflicting_company_numbers)
    ]
    .drop_duplicates(
        subset="CompanyNumber",
        keep="first",
    )
    .copy()
)

print(
    "Rows after resolving company duplicates:",
    f"{len(companies_house_companies):,}",
)

Companies with conflicting labels: 1432
Rows after resolving company duplicates: 49,591


# Standardising company names

The same cleaning rules applied to the OSM and FHRS names are applied to this dataset.

In [14]:
def clean_business_names(names):
    return (names
        .astype("string")
        .str.casefold()
        .str.replace("&", " and ", regex=False)
        .str.replace("_", " ", regex=False)
        .str.replace(r"[^\w\s]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


companies_house_companies["CleanBusinessName"] = clean_business_names(companies_house_companies["CompanyName"])

companies_house_companies["CleanBusinessName"] = (companies_house_companies["CleanBusinessName"].replace("", pd.NA))

companies_house_companies = (companies_house_companies.dropna(subset="CleanBusinessName").copy())

companies_house_companies[["CompanyName", "CleanBusinessName", "BakeryLabel"]].sample(5)

,CompanyName,CleanBusinessName,BakeryLabel
51653,XIANSHU 168 LIMITED,xianshu 168 limited,0
54939,SPICY SPOT LTD,spicy spot ltd,0
10619,PANDERMA FOODS LTD,panderma foods ltd,0
54340,STONE TOP LEISURE LIMITED,stone top leisure limited,0
29350,EMNOURISH UK LTD,emnourish uk ltd,0


# Creating one record per business name

There is a chance that different companies end up with the same cleaned business name. Business names with conflicting labels after cleaning are removed and duplicates are reduced to one row so that company names do not receive disproportionate influence in model training.

In [19]:
name_label_counts = (companies_house_companies.groupby("CleanBusinessName")["BakeryLabel"].nunique())

conflicting_names = (name_label_counts[name_label_counts > 1].index)

print("Cleaned names with conflicting labels:", len(conflicting_names))

companies_house_training = (companies_house_companies[ ~companies_house_companies["CleanBusinessName"].isin(conflicting_names)]
    .drop_duplicates(subset="CleanBusinessName", keep="first")
    [["CompanyName", "CleanBusinessName", "BakeryLabel"]]
    .rename(columns={"CompanyName": "DisplayName"})
    .reset_index(drop=True))

print(f"Unique usable names: {len(companies_house_training)}")

print(f"\nTraining label counts:\n {companies_house_training["BakeryLabel"].value_counts().rename({0: "Non-bakery", 1: "Bakery",})}")

companies_house_training.sample(5)

Cleaned names with conflicting labels: 0
Unique usable names: 49591

Training label counts:
 BakeryLabel
Non-bakery    43135
Bakery         6456
Name: count, dtype: int64


,DisplayName,CleanBusinessName,BakeryLabel
15137,AHMED HALAL MEAT-UK LTD,ahmed halal meat uk ltd,0
25188,MINI PUNJAB LTD,mini punjab ltd,0
3371,JAKE'S BAKE LIMITED,jake s bake limited,1
35778,NUESTRA FAMILIA RESTAURANTS LTD,nuestra familia restaurants ltd,0
40825,OCEAN PANASIA LTD,ocean panasia ltd,0


# Validation on Company House dataset

The final datset is checked for missing values, duplicates and valid labels. The label count is also checked to ensure that the numbers add up correctly.

In [20]:
print(f"Training rows: {len(companies_house_training)}")

print(f"\nMissing display names: {companies_house_training["DisplayName"].isna().sum()}")
print(f"Missing cleaned names: {companies_house_training["CleanBusinessName"].isna().sum()}")
print(f"Duplicate cleaned names: {companies_house_training["CleanBusinessName"].duplicated().sum()}")
print(f"Missing labels: {companies_house_training["BakeryLabel"].isna().sum()}")

print(f"Valid labels: {(companies_house_training["BakeryLabel"].isin([0, 1])).sum()}")

print(f"\nFinal label counts: {companies_house_training["BakeryLabel"].value_counts().rename({0: "Non-bakery", 1: "Bakery"})}")

Training rows: 49591

Missing display names: 0
Missing cleaned names: 0
Duplicate cleaned names: 0
Missing labels: 0
Valid labels: 49591

Final label counts: BakeryLabel
Non-bakery    43135
Bakery         6456
Name: count, dtype: int64


# Saving validated dataset

In [18]:
companies_house_training.to_csv(COMPANIES_HOUSE_TRAINING_DATA_PATH, index=False)

print(f"Prepared Companies House training data saved to:\n {COMPANIES_HOUSE_TRAINING_DATA_PATH}")

Prepared Companies House training data saved to:
 ..\data\interim\business\companies_house_training_names_2026-07-28.csv
